# 4.1 Diseño experimental

En esta sección definimos el experimento que compara los tres modelos base: variables, régimen de entrenamiento, métricas, umbrales de decisión, número de réplicas y controles de reproducibilidad. Todas estas decisiones quedaron registradas en un preregistro fechado antes de entrenar.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src import config as C
from src import plots as P
from src.persist import is_done, load_pickle, save_pickle

P.setup()
C.ensure_dirs()
pd.set_option("display.precision", 4)
print("Resultados:", C.RESULTS.name, "| modelos:", ", ".join(C.MODELS))

Resultados: results | modelos: CNN, MobileNetV2, ResNet-18


## Variables del experimento

| Elemento | Definición |
|---|---|
| Entrada | Tensor `float32` de forma $(3, 224, 224)$, normalizado por canal con estadísticas de entrenamiento |
| Salida | Etiqueta $y \in \{0,\dots,5\}$ (*cardboard, glass, metal, paper, plastic, trash*) |
| Factor | Arquitectura: CNN estilo VGG reducida, MobileNetV2, ResNet-18 |
| Réplica | Semilla, que controla la inicialización, el orden de los lotes, el aumento de datos y el dropout |
| Bloque | Semilla: los tres modelos comparten partición y semillas, por lo que las métricas están pareadas |
| Respuesta primaria | F1 macro en el conjunto de prueba |

**Régimen de entrenamiento.** Entrenamos los tres modelos desde cero (`weights=None`). La CNN no dispone de pesos preentrenados; preentrenar solo MobileNetV2 y ResNet-18 confundiría el efecto de la arquitectura con el de la transferencia desde ImageNet.

**Relación con la regresión logística.** Los tres modelos terminan en una capa lineal con *softmax*:

$$p(y=k\mid x)=\frac{\exp\big(w_k^\top \phi_\theta(x)+b_k\big)}{\sum_{j=1}^{6}\exp\big(w_j^\top \phi_\theta(x)+b_j\big)},$$

es decir, una regresión logística multinomial sobre rasgos $\phi_\theta(x)$ aprendidos por la red. Los modelos difieren en $\phi_\theta$, es decir, en su sesgo inductivo (sección 3, §9).

**Métricas.** La métrica primaria es el F1 macro, debido al desbalance de clases (sección 2.2). Las secundarias son accuracy, balanced accuracy, F1 ponderado y por clase, precisión y recall macro, AUC-ROC y AUC-PR macro (uno contra resto), coeficiente de correlación de Matthews (MCC) y error de calibración esperado (ECE).

**Umbrales.** Nivel de significancia $\alpha = 0.05$, potencia objetivo de 0.80 y diferencia mínima de relevancia práctica $\delta = 0.02$ en F1 macro.

**Secuencia.** (1) Búsqueda de hiperparámetros con el mismo espacio y presupuesto para los tres modelos, sin usar el conjunto de prueba (sección 4.3). (2) Estudio piloto con 5 semillas y la configuración elegida (sección 4.4). (3) Cálculo del número de semillas $R$. (4) Corridas hasta completar $R$ (sección 4.5). Como el piloto usa la configuración definitiva, sus 5 corridas forman parte del experimento final y la varianza estimada corresponde a la del experimento real.

**Reproducibilidad.** Fijamos las semillas de `random`, `numpy` y `torch` (CPU y CUDA), activamos `cudnn.deterministic`, desactivamos `cudnn.benchmark` y usamos `torch.use_deterministic_algorithms(True, warn_only=True)`. El orden de los lotes y el aumento de cada imagen dependen solo de la semilla y de la época. Algunas operaciones de cuDNN no tienen versión determinista y el entrenamiento en precisión mixta introduce redondeos, por lo que puede existir una variación residual pequeña entre ejecuciones de una misma semilla en GPU. Al final de cada época guardamos un punto de control completo, que permite reanudar una corrida interrumpida sin alterar su resultado.

## Modelos base

In [2]:
from src.models import build_model, model_info

rows = []
for m in C.MODELS:
    info = model_info(build_model(m), m)
    rows.append({"modelo": m, "parámetros": info["params_total"], "tamaño (MB)": round(info["size_mb"], 2),
                 "GMAC por imagen": round(info["gmacs"], 2)})
tabla_modelos = pd.DataFrame(rows).set_index("modelo")
save_pickle(tabla_modelos, C.SUMMARY / "tabla_modelos.pkl")
tabla_modelos.to_csv(C.SUMMARY / "tabla_modelos.csv")
display(tabla_modelos)

,parámetros,tamaño (MB),GMAC por imagen
modelo,,,
CNN,1174758,4.49,2.59
MobileNetV2,2231558,8.64,0.30
ResNet-18,11179590,42.68,1.81


| Modelo | Configuración | Sesgo inductivo |
|---|---|---|
| CNN | 4 bloques [Conv3×3, BN, ReLU, Conv3×3, BN, ReLU, MaxPool2] con 32-64-128-256 canales, *global average pooling*, dropout y Linear(256→6). Sin atajos | Localidad y jerarquía de rasgos |
| MobileNetV2 | `torchvision.models.mobilenet_v2(weights=None, num_classes=6, width_mult=1.0)` | Convolución separable en profundidad y cuellos de botella lineales |
| ResNet-18 | `torchvision.models.resnet18(weights=None, num_classes=6)`, con dropout antes de la capa final | Aprendizaje residual con atajos identidad |

El dropout de la capa final es un hiperparámetro común a los tres modelos y forma parte de la búsqueda de la sección 4.3.

## Entorno y preregistro

In [3]:
from src.design import write_preregistro, write_requirements_lock
from src.evaluate import env_info
from src.seeds import configure_determinism

env = env_info(configure_determinism())
save_pickle(env, C.COMMON / "env.pkl")
display(pd.Series({k: v for k, v in env.items() if k != "determinism"}, dtype=object).to_frame("valor"))
write_requirements_lock(ROOT / "requirements-lock.txt")
pre = write_preregistro()          # si ya existe, se conserva el original (fecha anterior a las corridas)
print("Preregistro:", pre.name)

,valor
python,3.12.10
torch,2.15.0+cu132
torchvision,0.29.2+cu132
cuda_available,True
cuda,13.2
cudnn,92600
gpu,NVIDIA GeForce RTX 4060 Laptop GPU
driver,592.82
cpu,"AMD64 Family 25 Model 117 Stepping 2, Authenti..."
cpu_count,16


Preregistro: preregistro_2026-11-08.md


In [4]:
display(Markdown(pre.read_text(encoding="utf-8")))

# Preregistro del experimento de modelos base

Fecha: 2026-11-08 02:12. Se escribe antes de la búsqueda de hiperparámetros y de las corridas definitivas.

## Hipótesis
- H0 global: las tres arquitecturas (CNN estilo VGG reducida, MobileNetV2 y ResNet-18), entrenadas desde cero con el mismo protocolo, tienen la misma distribución de F1 macro en prueba.
- H1 global: al menos una difiere.
- Por pares: H0: la diferencia media de F1 macro pareada por semilla es 0.

## Métrica, umbrales y unidad de réplica
- Métrica primaria: F1 macro en el conjunto de prueba. Secundarias: accuracy, balanced accuracy, F1 ponderado y por clase, precisión y recall macro, AUC-ROC macro (OvR), AUC-PR macro, MCC y ECE.
- alpha = 0.05; potencia objetivo = 0.8; delta (relevancia práctica) = 0.02 en F1 macro.
- Unidad de réplica: la semilla (inicialización, orden de lotes, aumentación y dropout). Las tres arquitecturas comparten partición y semillas: datos pareados por semilla.

## Datos
- Partición v2 del EDA (`trashnet_split_v2_s1234.csv`): 1800/360/361, estratificada y agrupada por casi-duplicados, sin las 6 imágenes con etiqueta contradictoria. Huella del test verificada antes de cada corrida.
- Entrada: 224x224, redimensionado directo (todas las imágenes son 512x384). Normalización con media y DE por canal de train. Pesos de clase con train.
- Aumentación solo en train: {'RandomResizedCrop': {'size': 224, 'scale': (0.7, 1.0)}, 'RandomHorizontalFlip': {'p': 0.5}, 'RandomVerticalFlip': {'p': 0.5}, 'RandomRotation': {'degrees': 15}, 'ColorJitter': {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2, 'hue': 0.02}}.

## Entrenamiento (igual para los tres)
- Régimen: desde cero (weights=None) para los tres. La CNN no tiene pesos preentrenados; preentrenar solo dos rompería la equidad. La transferencia desde ImageNet queda para el Entregable 2.
- Valores por defecto: {'optimizer': 'AdamW', 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.3, 'batch_size': 32, 'max_epochs': 100, 'warmup_epochs': 3, 'eta_min_factor': 0.01, 'patience': 15, 'min_delta': 0.0001, 'grad_clip': 5.0, 'amp': True, 'class_weighted_loss': True}.
- Búsqueda de hiperparámetros: malla de 8 configuraciones, idéntica para los tres modelos, semilla 100, máximo 50 épocas, selección por F1 macro de validación. El test no se usa.

| trial | lr | weight_decay | dropout |
|---|---|---|---|
| 0 | 0.001 | 0.0001 | 0.2 |
| 1 | 0.001 | 0.0001 | 0.4 |
| 2 | 0.001 | 0.0005 | 0.2 |
| 3 | 0.001 | 0.0005 | 0.4 |
| 4 | 0.0003 | 0.0001 | 0.2 |
| 5 | 0.0003 | 0.0001 | 0.4 |
| 6 | 0.0003 | 0.0005 | 0.2 |
| 7 | 0.0003 | 0.0005 | 0.4 |

## Tamaño de muestra (piloto interno)
- Orden: búsqueda de hiperparámetros, luego piloto con las semillas [0, 1, 2, 3, 4] y la configuración elegida. Las corridas del piloto son las primeras corridas definitivas (misma configuración).
- R = (z_(1-alpha/2) + z_(1-beta))^2 * sd_d^2 / delta^2 por par de modelos, con sd_d de las diferencias pareadas de F1 macro de validación del piloto. Se toma el máximo y se redondea hacia arriba.
- R_final = min(max(R, 6), 10). El mínimo de 6 permite pruebas por rangos; el máximo de 10 lo impone el tiempo de GPU. Si R supera 10, se reporta la diferencia mínima detectable (MDE) como limitación.

## Análisis planeado
- Global: Friedman (bloques = semillas, tratamientos = 3 modelos) con p exacto por permutación y W de Kendall.
- Pares (3): t pareada y Wilcoxon de rangos con signo; diferencia media con IC 95 % (t y bootstrap); d_z y r rango-biserial; p sin ajustar y ajustados por Holm. Nemenyi y Conover como complemento.
- Por imagen: McNemar por par y semilla (exacto si b + c < 25), con Holm dentro de cada semilla.
- AUC: bootstrap estratificado pareado sobre las imágenes de prueba (10000 remuestreos).
- Una diferencia es relevante si |diferencia media| >= delta y significativa tras Holm.
- Robustez: ruido gaussiano (0.05 y 0.10), desenfoque, brillo +-30 %, oclusión y fondo sustituido (hallazgo del EDA, 2.7).


## Conclusión

Comparamos tres arquitecturas entrenadas desde cero con un protocolo común. La semilla es la unidad de réplica y de bloqueo, y la métrica primaria es el F1 macro en prueba. Los umbrales, la regla para calcular $R$ y las pruebas estadísticas se fijaron antes del entrenamiento.